# 04 Modeling

ขั้นตอนนี้สร้างค่าทำนายของทุกโมเดลตาม Scope หัวข้อ 11–14 ด้วย expanding-window rolling origin ครอบคลุม 3 กลุ่มสินค้า, h = 2–6 และ 5 feature sets

**ขั้นตอนโดยสรุป**
- **origin:** ทุกแถว t ตั้งแต่ 2015-01 เป็นหนึ่ง origin โดย forecast origin คือวันที่ 15 ของ t+2
  - validation คือ t ช่วง 2015-01 ถึง 2018-12
  - test คือ t ตั้งแต่ 2019-01
- **training set ของแต่ละ origin:** แถวของกลุ่มและ horizon เดียวกันที่ `target_available_date ≤ forecast_origin` ซึ่งเทียบเท่ากับ t′ ≤ t − h
- **การ fit:** scaler และ model fit ใหม่ทุก origin แล้วทำนายแถว t แถวเดียว
- **การเลือก alpha ของ Ridge:** ใช้ MAE ช่วง validation อย่างเดียว แล้วจึงรันช่วง test ด้วย alpha ที่ตรึงไว้
- **XGBoost และ ARIMA:** XGBoost ใช้ค่าคงที่ ARIMA เลือก order ด้วย AIC จากข้อมูลถึง 2014-12

notebook นี้**ไม่คำนวณ metric ของช่วง test** การประเมินผลทั้งหมดอยู่ใน 05 Evaluation

In [1]:
from __future__ import annotations
import hashlib, itertools, json, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn, statsmodels, xgboost
from IPython import get_ipython
from IPython.display import Pretty, display
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA
from tqdm.std import tqdm as _tqdm
from xgboost import XGBRegressor

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'PROJECT_SCOPE.md').exists())
RAW_DIR, PROCESSED_DIR, METRICS_DIR = ROOT / 'data/raw', ROOT / 'data/processed', ROOT / 'logs/metrics'
VALIDATION_START, TEST_START = pd.Timestamp('2015-01-01'), pd.Timestamp('2019-01-01')
ARIMA_SELECTION_END = pd.Timestamp('2014-12-01')
ALPHAS = np.logspace(-2, 5, 15)
XGB_PARAMS = dict(n_estimators=200, max_depth=2, learning_rate=0.05, subsample=1.0,
                  random_state=0, n_jobs=1, tree_method='hist')
HIGH_RISK_QUANTILE = 0.80

class _DisplayLine:
    """ปลายทางของ tqdm ที่อัปเดต output ชิ้นเดิมแทนการพิมพ์บรรทัดใหม่"""
    def __init__(self):
        self.handle = display(Pretty(''), display_id=True)
    def write(self, text):
        text = text.replace('\r', '').strip()
        if text:
            self.handle.update(Pretty(text))
    def flush(self):
        pass

def tqdm(*args, **kwargs):
    # ใน notebook ให้ bar อัปเดตใน output เดิม (ไม่ใช้ widget จึงแสดงได้ทั้งตอนรันสดและในไฟล์ที่บันทึก)
    # นอก notebook ให้เขียนลง stdout ตามปกติ
    in_kernel = get_ipython() is not None and hasattr(get_ipython(), 'kernel')
    kwargs.setdefault('file', _DisplayLine() if in_kernel else sys.stdout)
    kwargs.setdefault('mininterval', 1.0)
    kwargs.setdefault('ncols', 110)
    return _tqdm(*args, **kwargs)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): digest.update(chunk)
    return digest.hexdigest()

manifest = pd.read_csv(METRICS_DIR / '03_processed_manifest.csv')
table_path = ROOT / manifest.loc[0, 'file']
assert sha256(table_path) == manifest.loc[0, 'sha256'], 'modeling_table.csv ไม่ตรงกับ manifest ของขั้น 03'
date_cols = ['month_ce', 'forecast_origin', 'target_month', 'target_available_date']
modeling = pd.read_csv(table_path, dtype={'commodity_code': str}, parse_dates=date_cols)
feature_sets = pd.read_csv(METRICS_DIR / '03_feature_sets.csv', dtype={'feature_set': str})
FEATURE_SETS = {row.feature_set: row.features.split('|') for row in feature_sets.itertuples()}
GROUPS = sorted(modeling.commodity_code.unique())
HORIZONS = sorted(int(h) for h in modeling.horizon.unique())

sources = json.loads((RAW_DIR / 'sources.json').read_text(encoding='utf-8'))['sources']
imi_source = next(x for x in reversed(sources) if x['source_id'].startswith('imi-monthly-'))
assert sha256(RAW_DIR / imi_source['file']) == imi_source['sha256']
raw = pd.read_csv(RAW_DIR / imi_source['file'], dtype={'commodityCode': str})
raw['commodityCode'] = raw.commodityCode.str.zfill(3)
raw['month_ce'] = pd.to_datetime(dict(year=raw.year - 543, month=raw.month, day=1))
log_imi = {g: np.log(raw[raw.commodityCode == g].set_index('month_ce')['index'].sort_index().asfreq('MS'))
           for g in GROUPS}

print(f'modeling_table: {len(modeling):,} แถว, กลุ่ม {GROUPS}, horizon {HORIZONS}')
print(f'feature sets: {list(FEATURE_SETS)}')
print(f'sklearn {sklearn.__version__}, statsmodels {statsmodels.__version__}, xgboost {xgboost.__version__}')

modeling_table: 4,560 แถว, กลุ่ม ['100', '318', '402'], horizon [2, 3, 4, 5, 6]
feature sets: ['1', '2', '3', '3-noRecent', '3-cutoff24']
sklearn 1.9.1, statsmodels 0.15.0, xgboost 3.4.1


## 1. Origin และ training set

ฟังก์ชัน `training_rows` เป็นจุดเดียวที่กำหนดว่าแถวใดใช้ train ได้ ณ origin หนึ่ง ทุกโมเดลใช้ฟังก์ชันนี้ร่วมกัน

ตารางนี้บันทึกค่าที่คำนวณจาก training set ของแต่ละ origin ไว้ใช้ในขั้น 05:
- threshold ของ High Risk (quantile 80% ของ y)
- สัดส่วนเดือนที่ y เป็นบวก ซึ่งใช้เป็น baseline ของ directional accuracy

In [2]:
panels = {(g, h): frame.sort_values('month_ce').reset_index(drop=True)
          for (g, h), frame in modeling.groupby(['commodity_code', 'horizon'])}

def training_rows(panel: pd.DataFrame, row: pd.Series) -> pd.DataFrame:
    train = panel[panel.target_available_date <= row.forecast_origin]
    assert train.month_ce.max() <= row.month_ce - pd.DateOffset(months=int(row.horizon))
    assert (train.month_ce < row.month_ce).all()
    return train

origin_rows = []
for (g, h), panel in tqdm(panels.items(), desc='สร้าง origin', unit='กลุ่ม×h'):
    for _, row in panel[panel.month_ce >= VALIDATION_START].iterrows():
        train = training_rows(panel, row)
        origin_rows.append({
            'commodity_code': g, 'horizon': h, 'month_ce': row.month_ce,
            'forecast_origin': row.forecast_origin,
            'split': 'validation' if row.month_ce < TEST_START else 'test',
            'y_true': row.target_y_pct, 'train_n': len(train),
            'train_last_t': train.month_ce.max(),
            'high_risk_threshold': float(np.quantile(train.target_y_pct, HIGH_RISK_QUANTILE)),
            'train_up_share': float((train.target_y_pct > 0).mean()),
        })
origins = pd.DataFrame(origin_rows)
origins.to_csv(METRICS_DIR / '04_origins.csv', index=False)
display(origins.groupby(['split', 'horizon']).agg(
    origins_per_group=('month_ce', lambda s: s.size // len(GROUPS)),
    first_t=('month_ce', 'min'), last_t=('month_ce', 'max'),
    min_train_n=('train_n', 'min'), max_train_n=('train_n', 'max')))
assert (origins.query("split == 'validation'").groupby(['commodity_code', 'horizon']).size() == 48).all()

สร้าง origin: 100%|######################################################| 15/15 [00:00<00:00, 16.46กลุ่ม×h/s]

origins_per_group    first_t     last_t  min_train_n  \
split      horizon                                                         
test       2                       90 2019-01-01 2026-06-01          215   
           3                       89 2019-01-01 2026-05-01          214   
           4                       88 2019-01-01 2026-04-01          213   
           5                       87 2019-01-01 2026-03-01          212   
           6                       86 2019-01-01 2026-02-01          211   
validation 2                       48 2015-01-01 2018-12-01          167   
           3                       48 2015-01-01 2018-12-01          166   
           4                       48 2015-01-01 2018-12-01          165   
           5                       48 2015-01-01 2018-12-01          164   
           6                       48 2015-01-01 2018-12-01          163   

                    max_train_n  
split      horizon               
test       2                304  
           3                302  
           4                300  
           5                298  
           6                296  
validation 2                214  
           3                213  
           4                212  
           5                211  
           6                210

## 2. Baselines

- **No-change:** ŷ = 0
- **Historical mean:** ŷ = ค่าเฉลี่ยของ y ใน training set ของ origin นั้น
- **Seasonal naive:** ŷ = 100 × [log IMI(t+h−12) − log IMI(t−12)] ใช้ IMI ไม่เกินเดือน t เพราะ h ≤ 12

In [3]:
prediction_parts = []
base_cols = ['commodity_code', 'horizon', 'month_ce', 'forecast_origin', 'split', 'y_true']

def add_predictions(frame: pd.DataFrame, model: str, feature_set: str, y_pred, alpha=np.nan):
    out = frame[base_cols].copy()
    out['model'], out['feature_set'], out['alpha'], out['y_pred'] = model, feature_set, alpha, np.asarray(y_pred, dtype=float)
    prediction_parts.append(out)

add_predictions(origins, 'no_change', 'none', np.zeros(len(origins)))
hist_mean = [panels[(r.commodity_code, r.horizon)].pipe(lambda p: training_rows(p, r)).target_y_pct.mean()
             for r in origins.itertuples()]
add_predictions(origins, 'historical_mean', 'none', hist_mean)
seasonal = []
for r in origins.itertuples():
    series, h = log_imi[r.commodity_code], int(r.horizon)
    base = r.month_ce - pd.DateOffset(months=12)
    assert base + pd.DateOffset(months=h) <= r.month_ce
    seasonal.append(100 * (series[base + pd.DateOffset(months=h)] - series[base]))
add_predictions(origins, 'seasonal_naive', 'none', seasonal)
print('baselines:', sum(len(p) for p in prediction_parts), 'ค่าทำนาย')

baselines: 6120 ค่าทำนาย


## 3. ARIMA

เลือก order ของ ARIMA(p, 1, q) บน log IMI ด้วย AIC จากข้อมูล 2000-01 ถึง 2014-12 แยกตามกลุ่ม
- grid: p, q ∈ {0, 1, 2} และ trend ∈ {ไม่มี drift, มี drift}
- ตรึง order ที่ได้ไว้ แล้ว fit coefficient ใหม่ทุก origin ด้วย log IMI ถึงเดือน t
- ARIMA ไม่ขึ้นกับ horizon จึง fit หนึ่งครั้งต่อ (กลุ่ม, t) แล้วพยากรณ์ 6 ขั้นใช้กับทุก h

In [4]:
def fit_arima(series, order, trend):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return ARIMA(series, order=order, trend=trend).fit()

order_rows = []
grid = list(itertools.product(GROUPS, range(3), range(3), ('n', 't')))
for g, p, q, trend in tqdm(grid, desc='ARIMA เลือก order', unit='fit'):
    series = log_imi[g][:ARIMA_SELECTION_END]
    try:
        result = fit_arima(series, (p, 1, q), trend)
        order_rows.append({'commodity_code': g, 'p': p, 'd': 1, 'q': q, 'trend': trend,
                           'aic': result.aic, 'converged': bool(result.mle_retvals.get('converged', True))})
    except Exception as error:
        order_rows.append({'commodity_code': g, 'p': p, 'd': 1, 'q': q, 'trend': trend,
                           'aic': np.nan, 'converged': False, 'error': str(error)[:80]})
arima_grid = pd.DataFrame(order_rows)
arima_selected = (arima_grid[arima_grid.converged].sort_values('aic')
                  .groupby('commodity_code').head(1).set_index('commodity_code'))
arima_grid.to_csv(METRICS_DIR / '04_arima_order_selection.csv', index=False)
display(arima_selected[['p', 'd', 'q', 'trend', 'aic']])

started = time.time()
arima_forecasts = {}
arima_jobs = [(g, t) for g in GROUPS for t in sorted(origins.loc[origins.commodity_code == g, 'month_ce'].unique())]
for g, t in tqdm(arima_jobs, desc='ARIMA rolling', unit='fit'):
    sel = arima_selected.loc[g]
    result = fit_arima(log_imi[g][:t], (int(sel.p), 1, int(sel.q)), sel.trend)
    forecast = np.asarray(result.forecast(steps=max(HORIZONS)))
    arima_forecasts[(g, t)] = 100 * (forecast - log_imi[g][t])
arima_pred = [arima_forecasts[(r.commodity_code, r.month_ce)][int(r.horizon) - 1] for r in origins.itertuples()]
add_predictions(origins, 'arima', 'none', arima_pred)
print(f'ARIMA: fit {len(arima_forecasts)} ครั้ง ใช้เวลา {time.time() - started:.0f} วินาที')

ARIMA เลือก order: 100%|#####################################################| 54/54 [00:01<00:00, 43.37fit/s]

,p,d,q,trend,aic
commodity_code,,,,,
402,2,1,2,n,-881.145242
318,1,1,2,n,-875.515598
100,1,1,0,n,-682.981728


ARIMA rolling: 100%|#######################################################| 414/414 [00:12<00:00, 34.28fit/s]

ARIMA: fit 414 ครั้ง ใช้เวลา 12 วินาที


## 4. Ridge Regression

สำหรับแต่ละ (กลุ่ม, h, feature set):
1. รันทุก alpha ใน grid 0.01–100,000 (15 ค่าแบบ log scale) บน validation origins
   - รอบแรกใช้ grid 0.01–1000 แล้วพบว่า alpha ที่เลือกชนขอบบน 31 จาก 75 ชุด จึงขยายขอบบน การตัดสินใจนี้ใช้ข้อมูล validation เท่านั้น
   - เมื่อ alpha สูงมาก Ridge จะเข้าใกล้ Historical mean ถ้ายังชนขอบบนอีก แปลว่า feature ช่วยได้น้อยในช่วง validation
2. เลือก alpha ที่ MAE ต่ำสุด ถ้าเท่ากันเลือก alpha ที่ใหญ่กว่า
3. ใช้ alpha นั้นกับ test origins

StandardScaler fit บน training set ของแต่ละ origin เท่านั้น

In [5]:
def fit_predict_ridge(train, row, features, alphas):
    scaler = StandardScaler().fit(train[features])
    x_train, x_row = scaler.transform(train[features]), scaler.transform(row[features].to_frame().T.astype(float))
    return [float(Ridge(alpha=a).fit(x_train, train.target_y_pct).predict(x_row)[0]) for a in alphas]

started = time.time()
alpha_rows = []
ridge_bar = tqdm(total=len(panels) * len(FEATURE_SETS), desc='Ridge', unit='ชุด')
for (g, h), panel in panels.items():
    val_rows = panel[(panel.month_ce >= VALIDATION_START) & (panel.month_ce < TEST_START)]
    test_rows = panel[panel.month_ce >= TEST_START]
    for fs, features in FEATURE_SETS.items():
        val_preds = np.array([fit_predict_ridge(training_rows(panel, r), r, features, ALPHAS)
                              for _, r in val_rows.iterrows()])
        val_mae = np.abs(val_preds - val_rows.target_y_pct.to_numpy()[:, None]).mean(axis=0)
        best = max(i for i in range(len(ALPHAS)) if np.isclose(val_mae[i], val_mae.min()))
        alpha = float(ALPHAS[best])
        alpha_rows += [{'commodity_code': g, 'horizon': h, 'feature_set': fs, 'alpha': a,
                        'validation_mae': m, 'selected': i == best} for i, (a, m) in enumerate(zip(ALPHAS, val_mae))]
        # test ใช้ alpha ที่ตรึงจาก validation เท่านั้น
        test_preds = [fit_predict_ridge(training_rows(panel, r), r, features, [alpha])[0] for _, r in test_rows.iterrows()]
        frame = origins[(origins.commodity_code == g) & (origins.horizon == h)].sort_values('month_ce')
        add_predictions(frame, 'ridge', fs, np.concatenate([val_preds[:, best], test_preds]), alpha)
        ridge_bar.set_postfix(group=g, h=h, fs=fs)
        ridge_bar.update(1)
ridge_bar.close()
ridge_alphas = pd.DataFrame(alpha_rows)
ridge_alphas.to_csv(METRICS_DIR / '04_ridge_alpha_selection.csv', index=False)
boundary = ridge_alphas[ridge_alphas.selected & np.isclose(ridge_alphas.alpha, ALPHAS.max())]
print(f'alpha ที่เลือกชนขอบบน {ALPHAS.max():g}: {len(boundary)} จาก {int(ridge_alphas.selected.sum())} ชุด')
display(ridge_alphas[ridge_alphas.selected].pivot_table(index=['commodity_code', 'horizon'], columns='feature_set', values='alpha'))
print(f'Ridge ใช้เวลา {time.time() - started:.0f} วินาที')

Ridge: 100%|##################################| 75/75 [00:34<00:00,  2.19ชุด/s, fs=3-cutoff24, group=402, h=6]

alpha ที่เลือกชนขอบบน 100000: 9 จาก 75 ชุด


feature_set                         1              2            3  \
commodity_code horizon                                              
100            2        100000.000000  100000.000000     0.010000   
               3        100000.000000  100000.000000    10.000000   
               4        100000.000000    1000.000000   100.000000   
               5        100000.000000     316.227766   316.227766   
               6        100000.000000     316.227766   316.227766   
318            2           316.227766     316.227766   100.000000   
               3           316.227766     316.227766   100.000000   
               4            31.622777     316.227766   100.000000   
               5             1.000000     316.227766   316.227766   
               6           316.227766    1000.000000   316.227766   
402            2           100.000000     316.227766  1000.000000   
               3           316.227766    1000.000000  1000.000000   
               4           316.227766    1000.000000  1000.000000   
               5           316.227766    1000.000000  1000.000000   
               6           316.227766    1000.000000  1000.000000   

feature_set              3-cutoff24     3-noRecent  
commodity_code horizon                              
100            2           0.100000  100000.000000  
               3          31.622777  100000.000000  
               4         100.000000    3162.277660  
               5         316.227766    3162.277660  
               6         316.227766    1000.000000  
318            2         100.000000     316.227766  
               3         100.000000     316.227766  
               4         100.000000     316.227766  
               5         316.227766     316.227766  
               6         316.227766     316.227766  
402            2        1000.000000     316.227766  
               3        1000.000000    1000.000000  
               4        1000.000000    1000.000000  
               5        1000.000000    1000.000000  
               6        1000.000000    1000.000000

Ridge ใช้เวลา 34 วินาที


## 5. XGBoost

ใช้ค่าคงที่ไม่ tune ได้แก่ 200 trees, learning_rate 0.05, max_depth 2 และ random_state 0
- เลือก max_depth 2 ซึ่งเป็นค่าล่างของช่วง 2–3 ใน Scope เพราะ training set มีเพียง 160–300 แถว
- รันทั้งช่วง validation และ test เพื่อให้ขั้น 05 เลือก "โมเดลที่ดีที่สุด" จาก validation ได้

In [6]:
started = time.time()
xgb_bar = tqdm(total=len(FEATURE_SETS) * sum(int((p.month_ce >= VALIDATION_START).sum()) for p in panels.values()),
               desc='XGBoost', unit='fit')
for (g, h), panel in panels.items():
    rows = panel[panel.month_ce >= VALIDATION_START]
    frame = origins[(origins.commodity_code == g) & (origins.horizon == h)].sort_values('month_ce')
    for fs, features in FEATURE_SETS.items():
        preds = []
        for _, r in rows.iterrows():
            train = training_rows(panel, r)
            model = XGBRegressor(**XGB_PARAMS).fit(train[features], train.target_y_pct)
            preds.append(float(model.predict(r[features].to_frame().T.astype(float))[0]))
            xgb_bar.update(1)
        xgb_bar.set_postfix(group=g, h=h, fs=fs)
        add_predictions(frame, 'xgboost', fs, preds)
xgb_bar.close()
print(f'XGBoost ใช้เวลา {time.time() - started:.0f} วินาที')

XGBoost: 100%|##########################| 10200/10200 [03:02<00:00, 55.83fit/s, fs=3-cutoff24, group=402, h=6]

XGBoost ใช้เวลา 183 วินาที


## 6. Checkpoint และบันทึกผล

เงื่อนไขผ่าน:
- ทุกคู่ (โมเดล, feature set) มีค่าทำนายครบทุก origin ของทุกกลุ่มและ horizon
- ไม่มีค่าว่าง
- ค่า y_true ตรงกับตาราง origin

ตารางด้านล่างแสดง MAE ช่วง **validation** เท่านั้น เพื่อดูภาพรวมและใช้เลือกโมเดล ส่วน metric ของช่วง test อยู่ในขั้น 05

In [7]:
predictions = pd.concat(prediction_parts, ignore_index=True)
predictions = predictions.merge(
    origins[['commodity_code', 'horizon', 'month_ce', 'high_risk_threshold', 'train_up_share', 'train_n']],
    on=['commodity_code', 'horizon', 'month_ce'], how='left', validate='many_to_one')
expected = origins.groupby(['commodity_code', 'horizon']).size()
counts = predictions.groupby(['model', 'feature_set', 'commodity_code', 'horizon']).size()
for (model, fs, g, h), n in counts.items():
    assert n == expected[(g, h)], (model, fs, g, h, n)
assert counts.reset_index()[['model', 'feature_set']].drop_duplicates().shape[0] == 4 + 2 * len(FEATURE_SETS)
assert predictions.y_pred.notna().all() and np.isfinite(predictions.y_pred).all()
assert not predictions.duplicated(['model', 'feature_set', 'commodity_code', 'horizon', 'month_ce']).any()

pred_path = METRICS_DIR / '04_predictions.csv'
predictions.to_csv(pred_path, index=False)
run_manifest = pd.DataFrame([{
    'file': str(pred_path.relative_to(ROOT)), 'rows': len(predictions), 'sha256': sha256(pred_path),
    'modeling_table_sha256': manifest.loc[0, 'sha256'], 'model_series': counts.reset_index()[['model', 'feature_set']].drop_duplicates().shape[0],
    'validation_origins_per_group_h': 48, 'xgb_params': json.dumps(XGB_PARAMS), 'alpha_grid': '|'.join(f'{a:g}' for a in ALPHAS),
    'sklearn': sklearn.__version__, 'statsmodels': statsmodels.__version__, 'xgboost': xgboost.__version__}])
run_manifest.to_csv(METRICS_DIR / '04_run_manifest.csv', index=False)

val = predictions[predictions.split == 'validation'].assign(abs_error=lambda d: (d.y_pred - d.y_true).abs())
val_mae = val.groupby(['commodity_code', 'horizon', 'model', 'feature_set']).abs_error.mean().rename('validation_mae').reset_index()
val_mae.to_csv(METRICS_DIR / '04_validation_mae.csv', index=False)
display(val_mae[val_mae.horizon == 3].assign(series=lambda d: d.model + ' / ' + d.feature_set)
        .pivot_table(index='series', columns='commodity_code', values='validation_mae').round(3))
display(run_manifest.T)
print('CHECKPOINT PASSED: ทุกโมเดลมีค่าทำนายครบทุก origin และบันทึกลง logs/metrics แล้ว')

commodity_code,100,318,402
series,,,
arima / none,10.051,2.570,1.158
historical_mean / none,9.613,2.948,1.303
no_change / none,9.789,2.797,0.989
ridge / 1,9.615,2.573,1.146
ridge / 2,9.614,2.635,1.154
ridge / 3,7.975,2.180,1.144
ridge / 3-cutoff24,8.361,2.216,1.142
ridge / 3-noRecent,9.613,2.390,1.159
seasonal_naive / none,12.679,3.489,1.450


,0
file,logs/metrics/04_predictions.csv
rows,28560
sha256,7fd851a34cf5c05fbc013e0d5d89df1bc2d8c840aead90...
modeling_table_sha256,c97d79ab0c1afade35d0f1bcf999d51b4c0b58c774048c...
model_series,14
validation_origins_per_group_h,48
xgb_params,"{""n_estimators"": 200, ""max_depth"": 2, ""learnin..."
alpha_grid,0.01|0.0316228|0.1|0.316228|1|3.16228|10|31.62...
sklearn,1.9.1
statsmodels,0.15.0


CHECKPOINT PASSED: ทุกโมเดลมีค่าทำนายครบทุก origin และบันทึกลง logs/metrics แล้ว


In [8]:
log_path = ROOT / 'logs/process_log.md'
log = log_path.read_text(encoding='utf-8')
marker = 'ขั้นตอน: 04 Modeling'
if marker not in log:
    orders = '; '.join(f"{g}: ARIMA({int(r.p)},1,{int(r.q)}) trend={r.trend}" for g, r in arima_selected.iterrows())
    sel = ridge_alphas[ridge_alphas.selected]
    entry = f"""

## [2026-10-05] {marker}
**สิ่งที่ทำ:** สร้างค่าทำนายด้วย expanding-window rolling origin ครอบคลุม 3 กลุ่มสินค้า h = 2–6 และ 5 feature sets
- โมเดล: No-change, Historical mean, Seasonal naive, ARIMA, Ridge และ XGBoost
- validation คือ t ช่วง 2015-01 ถึง 2018-12 (48 origins) และ test คือ t ตั้งแต่ 2019-01
- training set ของทุก origin ใช้กฎ target_available_date ≤ forecast_origin ผ่านฟังก์ชันเดียวกัน

**ผลที่ได้:**
- ค่าทำนายรวม {len(predictions):,} แถว จาก {run_manifest.loc[0, 'model_series']} ชุด (โมเดล × feature set) ทุกชุดครบทุก origin
- ARIMA order ที่เลือกด้วย AIC จากข้อมูลถึง 2014-12: {orders}
- alpha ของ Ridge ที่เลือกจาก validation มีค่ากลาง {sel.alpha.median():g} (ช่วง {sel.alpha.min():g}–{sel.alpha.max():g}) และชนขอบบนของ grid {len(boundary)} จาก {len(sel)} ชุด
- ไฟล์ที่บันทึก: logs/metrics/04_predictions.csv, 04_origins.csv, 04_ridge_alpha_selection.csv, 04_arima_order_selection.csv, 04_validation_mae.csv และ 04_run_manifest.csv

**สิ่งที่พบ / ปัญหา:**
- รอบแรกใช้ alpha grid 0.01–1000 แล้วพบว่า alpha ที่เลือกชนขอบบน 31 จาก 75 ชุด จึงขยาย grid เป็น 0.01–100,000 และรันใหม่ทั้งขั้น การปรับนี้ใช้ข้อมูล validation เท่านั้น
- ARIMA ใน statsmodels 0.15 ไม่รับ steps ที่เป็น numpy integer จึงแปลง horizon เป็น int
- ไม่พบค่าทำนายว่างหรือ origin ที่ขาด ขั้นนี้ยังไม่ได้คำนวณ metric ของช่วง test

**การตัดสินใจและเหตุผล:**
- XGBoost ใช้ max_depth 2 ซึ่งเป็นค่าล่างของช่วงใน Scope เพราะ training set เล็ก
- threshold ของ High Risk และสัดส่วนเดือนขาขึ้นคำนวณจาก training set ของแต่ละ origin และบันทึกคู่กับค่าทำนาย เพื่อใช้ในขั้น 05 โดยไม่ต้องคำนวณใหม่

**ขั้นต่อไป:** 05 Evaluation
- เลือกโมเดลที่ดีที่สุดจาก validation MAE
- คำนวณ MAE, RMSE และ DA ช่วง test
- DM-HLN สำหรับการเปรียบเทียบหลัก 3 ข้อพร้อม Holm
- alert metrics, sensitivity cutoff24 และเส้นความแม่นตาม horizon (RQ4ก)
"""
    log_path.write_text(log.rstrip() + entry + '\n', encoding='utf-8')
    print('บันทึกผลลง logs/process_log.md แล้ว')
else:
    print('มีรายการ 04 Modeling ใน process log แล้ว จึงไม่บันทึกซ้ำ')

มีรายการ 04 Modeling ใน process log แล้ว จึงไม่บันทึกซ้ำ
